# SPLADE ablations and three-channel RRF

Этот notebook принимает широкие rankings из Kaggle и отвечает на один
заранее сформулированный вопрос: даёт ли русский learned-sparse retriever
дополнительный Recall@50 поверх уже выбранного BM25+dense RRF.

Методология защищает от подглядывания в test:

- используется тот же детерминированный 50/50 stratified dev/test split;
- pruning, local weight и финальный RRF grid выбираются только по dev;
- после первого test-look обнаруженная ещё на dev неполная factorial-сетка
  дополнена ячейкой q32/d192; для второго и последнего primary-look применяется
  Bonferroni alpha=0.025 и 97.5% bootstrap CI;
- primary endpoint — `BM25+dense+SPLADE` против зафиксированного
  `BM25+dense`; остальные тесты считаются secondary ablations;
- для secondary family дополнительно приводится Holm-adjusted p-value.

In [1]:
from pathlib import Path
from dataclasses import asdict
import json, re, sys

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from avito_retrieval.fusion import reciprocal_rank_fusion
from avito_retrieval.metrics import recall_at_k
from avito_retrieval.statistics import paired_recall_test

SPLADE_DIR = ROOT / "artifacts/splade_kaggle"
DENSE_DIR = ROOT / "artifacts/dense_kaggle"
RANK_DIR = ROOT / "artifacts/rankings"


def parse_rank(value: object) -> list[str]:
    return [] if value is None or pd.isna(value) or not str(value) else str(value).split()


def subset(mapping: dict, ids: set[str]) -> dict:
    return {query_id: mapping[query_id] for query_id in ids}


def holm_adjust(p_values: dict[str, float]) -> dict[str, float]:
    """Holm family-wise correction for exploratory engineering ablations."""
    ordered = sorted(p_values, key=p_values.get)
    adjusted: dict[str, float] = {}
    running = 0.0
    count = len(ordered)
    for index, name in enumerate(ordered):
        running = max(running, min(1.0, (count - index) * p_values[name]))
        adjusted[name] = running
    return adjusted

## 1. Контракт артефактов

Проверяем не только число строк, но и query coverage, отсутствие повторов,
существование каждого `item_id` в benchmark corpus и полноту всех восьми
ranking columns. Ошибка здесь делает дальнейшую метрику недействительной.

In [2]:
manifest = pd.read_parquet(ROOT / "artifacts/validation/manifest.parquet").copy()
labels = pd.read_parquet(ROOT / "artifacts/validation/labels.parquet")
benchmark = pd.read_parquet(ROOT / "dataset/benchmark_queries.parquet")
items = pd.read_parquet(ROOT / "dataset/benchmark_items.parquet", columns=["item_id"])
valid_items = set(items.item_id.astype(str))
relevant = labels.groupby("eval_query_id").item_id.agg(set).to_dict()

splade = pd.read_parquet(SPLADE_DIR / "splade_rankings.parquet")
splade_run = json.loads((SPLADE_DIR / "splade_run.json").read_text())
dense = pd.read_parquet(DENSE_DIR / "dense_rankings.parquet")
bm25 = pd.read_parquet(RANK_DIR / "bm25_rankings.parquet")

prefixes = [
    "splade_q32_d64", "splade_q64_d64",
    "splade_q32_d192", "splade_q64_d192",
    "splade_first_q32_d192",
]
rank_columns = [f"{prefix}_{scope}" for prefix in prefixes for scope in ("global", "local")]
assert set(rank_columns) <= set(splade.columns)
assert len(splade) == len(manifest) + len(benchmark)
assert splade.query_key.is_unique
assert set(splade[splade.split.eq("validation")].query_key) == set(manifest.eval_query_id)
assert set(splade[splade.split.eq("benchmark")].query_key) == set(benchmark.query_id)
for column in rank_columns:
    for ranking in splade[column].map(parse_rank):
        assert len(ranking) == len(set(ranking))
        assert len(ranking) <= 250
        assert set(ranking) <= valid_items
display(splade_run)
print("SPLADE artifacts valid:", splade.shape)

{'model': 'naver/neuclir22-splade-ru',
 'license': 'cc-by-nc-sa-4.0',
 'items': 189212,
 'passages': 554920,
 'queries': 4904,
 'chunk_words': 140,
 'overlap': 30,
 'max_chunks': 4,
 'max_length': 256,
 'document_max_dims': 192,
 'query_max_dims': 64,
 'tokenizer_unk_rate': 0.0,
 'gpus': ['Tesla T4', 'Tesla T4']}

SPLADE artifacts valid: (4904, 12)


## 2. Неизменный dev/test split

`primary_stratum = has_filters × local_pool_bucket`. Random seed и split
совпадают с dense notebook, поэтому мы не создаём новый удобный test после
просмотра предыдущих результатов.

In [3]:
manifest["fold"] = -1
splitter = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
for fold, (_, indices) in enumerate(splitter.split(manifest, manifest.primary_stratum)):
    manifest.loc[indices, "fold"] = fold
fold_by_id = manifest.set_index("eval_query_id").fold.to_dict()
dev_ids = {query_id for query_id, fold in fold_by_id.items() if fold == 0}
test_ids = set(fold_by_id) - dev_ids
display(pd.crosstab(manifest.primary_stratum, manifest.fold, normalize="index").round(3))

fold,0,1
primary_stratum,,
False|0,0.500,0.500
False|1-99,0.517,0.483
False|100-499,0.500,0.500
False|10000+,0.499,0.501
False|2000-9999,0.500,0.500
False|500-1999,0.500,0.500
True|0,0.503,0.497
True|1-99,0.462,0.538
True|100-499,0.506,0.494


## 3. Pruning, chunking и local-channel

Для чистых secondary ablations global/local объединяются с одинаковым весом
1.0. Затем отдельно выбираем production local weight по dev. Так изменение
pruning не смешивается с изменением географического веса. Бюджеты q32/q64
и d64/d192 заранее образуют компактную факторную сетку: короткому запросу
нужно меньше термов, чем документу, а четыре factorial-ячейки позволяют
разделить два эффекта без перебора десятков конфигураций по holdout. Первый
dev-прогон также упёрся в верхнюю границу local-weight=2.0, поэтому сетка
однократно расширена до 3.0 и 4.0 исключительно по dev. Четыре chunks
против первого проверяются снова, потому что статистический вывод dense
retriever не обязан переноситься на learned-sparse модель.

In [4]:
validation = splade[splade.split.eq("validation")].set_index("query_key")


def splade_predictions(prefix: str, local_weight: float = 1.0, top_k: int = 250) -> dict[str, list[str]]:
    return {
        query_id: reciprocal_rank_fusion(
            [parse_rank(row[f"{prefix}_global"]), parse_rank(row[f"{prefix}_local"])],
            weights=[1.0, local_weight],
            rrf_k=60,
            top_k=top_k,
        )
        for query_id, row in validation.iterrows()
    }


controlled = {prefix: splade_predictions(prefix, local_weight=1.0) for prefix in prefixes}
controlled_scores = pd.DataFrame(
    [
        {
            "variant": prefix,
            "dev": recall_at_k(subset(prediction, dev_ids), subset(relevant, dev_ids)),
        }
        for prefix, prediction in controlled.items()
    ]
)
display(controlled_scores.sort_values("dev", ascending=False))

local_grid = []
local_cache = {}
for prefix in prefixes:
    for local_weight in [0.75, 1.0, 1.15, 1.25, 1.5, 2.0, 3.0, 4.0]:
        prediction = splade_predictions(prefix, local_weight=local_weight)
        local_cache[(prefix, local_weight)] = prediction
        local_grid.append(
            {
                "prefix": prefix,
                "local_weight": local_weight,
                "dev_recall": recall_at_k(
                    subset(prediction, dev_ids), subset(relevant, dev_ids)
                ),
            }
        )
local_grid = pd.DataFrame(local_grid).sort_values(
    ["dev_recall", "local_weight"], ascending=[False, True]
)
selected_prefix = str(local_grid.iloc[0].prefix)
selected_local_weight = float(local_grid.iloc[0].local_weight)
splade_pred = local_cache[(selected_prefix, selected_local_weight)]
display(local_grid.head(12))
print("Selected only on dev:", selected_prefix, "local_weight=", selected_local_weight)

,variant,dev
2,splade_q32_d192,0.551305
3,splade_q64_d192,0.519929
0,splade_q32_d64,0.509611
1,splade_q64_d64,0.479024
4,splade_first_q32_d192,0.458102


,prefix,local_weight,dev_recall
21,splade_q32_d192,2.00,0.633361
22,splade_q32_d192,3.00,0.633361
23,splade_q32_d192,4.00,0.633361
20,splade_q32_d192,1.50,0.618108
29,splade_q64_d192,2.00,0.597268
30,splade_q64_d192,3.00,0.597268
31,splade_q64_d192,4.00,0.597268
19,splade_q32_d192,1.25,0.593719
5,splade_q32_d64,2.00,0.587765
6,splade_q32_d64,3.00,0.587765


Selected only on dev: splade_q32_d192 local_weight= 2.0


## 4. Трёхканальный RRF

BM25 и dense rankings остаются ровно такими, какими были выбраны в notebook
05. Для SPLADE перебирается маленькая заранее заданная сетка веса и `rrf_k`
только на dev. Значения `rrf_k={10,20,40,60,100}` покрывают ранний и плавный
режимы затухания, а веса 0.25–2.0 допускают как вспомогательный, так и равный
вклад канала. Более широкая сетка увеличила бы selection bias при 35 точках.
Мы не смешиваем исходные BM25, cosine и SPLADE scores: RRF использует только
ранги и поэтому не требует некорректной общей калибровки.

In [5]:
joined = (
    splade.merge(dense, on=["query_key", "split"], validate="one_to_one")
    .merge(bm25, on=["query_key", "split"], validate="one_to_one")
)
joined_validation = joined[joined.split.eq("validation")].set_index("query_key")

bm25_pred = {query_id: parse_rank(row.bm25) for query_id, row in joined_validation.iterrows()}
dense_pred = {
    query_id: reciprocal_rank_fusion(
        [parse_rank(row.dense_global), parse_rank(row.dense_local)],
        weights=[1.0, 1.15],
        top_k=250,
    )
    for query_id, row in joined_validation.iterrows()
}
baseline_pred = {
    query_id: reciprocal_rank_fusion(
        [bm25_pred[query_id], dense_pred[query_id]],
        weights=[1.0, 1.25],
        rrf_k=20,
        top_k=50,
    )
    for query_id in relevant
}

fusion_grid = []
fusion_cache = {}
for rrf_k in [10, 20, 40, 60, 100]:
    for splade_weight in [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]:
        prediction = {
            query_id: reciprocal_rank_fusion(
                [bm25_pred[query_id], dense_pred[query_id], splade_pred[query_id]],
                weights=[1.0, 1.25, splade_weight],
                rrf_k=rrf_k,
                top_k=50,
            )
            for query_id in relevant
        }
        fusion_cache[(rrf_k, splade_weight)] = prediction
        fusion_grid.append(
            {
                "rrf_k": rrf_k,
                "splade_weight": splade_weight,
                "dev_recall": recall_at_k(
                    subset(prediction, dev_ids), subset(relevant, dev_ids)
                ),
            }
        )
fusion_grid = pd.DataFrame(fusion_grid).sort_values(
    ["dev_recall", "splade_weight"], ascending=[False, True]
)
best = fusion_grid.iloc[0]
best_key = (int(best.rrf_k), float(best.splade_weight))
hybrid3_pred = fusion_cache[best_key]
display(fusion_grid.head(12))

comparison = pd.DataFrame(
    [
        {
            "method": "BM25+dense",
            "dev": recall_at_k(subset(baseline_pred, dev_ids), subset(relevant, dev_ids)),
            "test": recall_at_k(subset(baseline_pred, test_ids), subset(relevant, test_ids)),
            "full": recall_at_k(baseline_pred, relevant),
        },
        {
            "method": "SPLADE",
            "dev": recall_at_k(subset(splade_pred, dev_ids), subset(relevant, dev_ids)),
            "test": recall_at_k(subset(splade_pred, test_ids), subset(relevant, test_ids)),
            "full": recall_at_k(splade_pred, relevant),
        },
        {
            "method": f"BM25+dense+SPLADE {best_key}",
            "dev": recall_at_k(subset(hybrid3_pred, dev_ids), subset(relevant, dev_ids)),
            "test": recall_at_k(subset(hybrid3_pred, test_ids), subset(relevant, test_ids)),
            "full": recall_at_k(hybrid3_pred, relevant),
        },
    ]
)
display(comparison)

,rrf_k,splade_weight,dev_recall
8,20,0.50,0.842713
15,40,0.50,0.839587
7,20,0.25,0.838363
29,100,0.50,0.838227
1,10,0.50,0.837955
14,40,0.25,0.837684
21,60,0.25,0.837684
16,40,0.75,0.837004
30,100,0.75,0.836732
23,60,0.75,0.836596


,method,dev,test,full
0,BM25+dense,0.831430,0.809203,0.820317
1,SPLADE,0.633361,0.641245,0.637303
2,"BM25+dense+SPLADE (20, 0.5)",0.842713,0.811990,0.827352


## 5. Парные тесты и multiple-comparison control

Primary test имеет заранее заданное направление: добавление SPLADE должно
повысить per-query Recall@50 относительно BM25+dense. Для secondary ablations
одновременно проверяются pruning, chunking, local-channel и SPLADE-only;
поэтому к их randomization p-values применяется Holm correction. Это второй
и последний primary-look, поэтому Bonferroni-критерий требует `p < 0.025`
и 97.5% bootstrap CI целиком выше нуля. Так повторный запуск не превращается
в бесплатную дополнительную попытку на том же test.

In [6]:
global_selected = {
    query_id: parse_rank(row[f"{selected_prefix}_global"])
    for query_id, row in validation.iterrows()
}
primary = asdict(
    paired_recall_test(
        hybrid3_pred, baseline_pred, relevant, query_ids=sorted(test_ids),
        confidence=0.975,
    )
)
secondary_pairs = {
    "query_32_vs_64_at_d192": (
        controlled["splade_q32_d192"], controlled["splade_q64_d192"]
    ),
    "document_192_vs_64": (
        controlled["splade_q32_d192"],
        controlled["splade_q32_d64"],
    ),
    "four_chunks_vs_first": (
        controlled["splade_q32_d192"],
        controlled["splade_first_q32_d192"],
    ),
    "local_vs_global": (splade_pred, global_selected),
    "splade_vs_bm25": (splade_pred, bm25_pred),
}
secondary = {
    name: asdict(paired_recall_test(a, b, relevant, query_ids=sorted(test_ids)))
    for name, (a, b) in secondary_pairs.items()
}
adjusted = holm_adjust(
    {name: result["p_value_greater"] for name, result in secondary.items()}
)
for name in secondary:
    secondary[name]["holm_p_value_greater"] = adjusted[name]
print("Primary RRF3 vs RRF2:")
display(pd.DataFrame({"RRF3_vs_RRF2": primary}).T)
print("Secondary engineering ablations:")
display(pd.DataFrame(secondary).T)

primary_confirmed = bool(
    primary["mean_delta"] > 0
    and primary["ci_low"] > 0
    and primary["p_value_greater"] < 0.025
)
print("SPLADE accepted by pre-registered primary criterion:", primary_confirmed)

Primary RRF3 vs RRF2:


,mean_delta,ci_low,ci_high,p_value_greater,n_queries
RRF3_vs_RRF2,0.002787,-0.006729,0.012235,0.265837,1226.0


Secondary engineering ablations:


,mean_delta,ci_low,ci_high,p_value_greater,n_queries,holm_p_value_greater
query_32_vs_64_at_d192,0.024157,0.011691,0.036909,0.00010,1226.0,0.00025
document_192_vs_64,0.059978,0.040701,0.079323,0.00005,1226.0,0.00025
four_chunks_vs_first,0.109095,0.083333,0.135060,0.00005,1226.0,0.00025
local_vs_global,0.480968,0.452556,0.508972,0.00005,1226.0,0.00025
splade_vs_bm25,-0.106620,-0.132654,-0.080614,1.00000,1226.0,1.00000


SPLADE accepted by pre-registered primary criterion: False


## 6. Slices, candidate-union diagnostic и submission

Slice-метрики проверяют, не куплен ли общий прирост ценой регрессии на
запросах с фильтрами или tail-сегментах. Candidate union снова не является
допустимым answer: это широкий диагностический пул. `answer_splade.csv`
создаётся всегда для аудита, но основной `answer.csv` заменяется только если
primary test подтверждён.

In [7]:
slice_rows = []
for column in ["has_filters", "local_pool_bucket", "query_frequency_bucket", "positive_bucket"]:
    for value, group in manifest.groupby(column):
        ids = set(group.eval_query_id)
        base = recall_at_k(subset(baseline_pred, ids), subset(relevant, ids))
        new = recall_at_k(subset(hybrid3_pred, ids), subset(relevant, ids))
        slice_rows.append(
            {
                "slice": column,
                "value": str(value),
                "queries": len(ids),
                "BM25+dense": base,
                "BM25+dense+SPLADE": new,
                "delta": new - base,
            }
        )
slices = pd.DataFrame(slice_rows)
display(slices.sort_values(["slice", "value"]))

union_pred = {
    query_id: list(
        dict.fromkeys([*bm25_pred[query_id], *dense_pred[query_id], *splade_pred[query_id]])
    )
    for query_id in test_ids
}
union_recall = recall_at_k(union_pred, subset(relevant, test_ids), k=750)
print("BM25+dense+SPLADE candidate-union Recall:", union_recall)

bench = joined[joined.split.eq("benchmark")].set_index("query_key")
answers = []
for query_id, row in bench.iterrows():
    dense_rank = reciprocal_rank_fusion(
        [parse_rank(row.dense_global), parse_rank(row.dense_local)],
        weights=[1.0, 1.15],
        top_k=250,
    )
    splade_rank = reciprocal_rank_fusion(
        [
            parse_rank(row[f"{selected_prefix}_global"]),
            parse_rank(row[f"{selected_prefix}_local"]),
        ],
        weights=[1.0, selected_local_weight],
        rrf_k=60,
        top_k=250,
    )
    final = reciprocal_rank_fusion(
        [parse_rank(row.bm25), dense_rank, splade_rank],
        weights=[1.0, 1.25, best_key[1]],
        rrf_k=best_key[0],
        top_k=50,
    )
    answers.append({"query_id": query_id, "answer": " ".join(final)})

answer = pd.DataFrame(answers)
assert list(answer.columns) == ["query_id", "answer"]
assert len(answer) == len(benchmark) and answer.query_id.is_unique
assert set(answer.query_id) == set(benchmark.query_id)
for value in answer.answer:
    ids = value.split()
    assert len(ids) == 50 and len(ids) == len(set(ids))
    assert set(ids) <= valid_items
    assert all(re.fullmatch(r"[0-9a-f]{16}", item_id) for item_id in ids)
answer.to_csv(ROOT / "answer_splade.csv", index=False, encoding="utf-8")
if primary_confirmed:
    answer.to_csv(ROOT / "answer.csv", index=False, encoding="utf-8")

metrics = {
    "model": splade_run["model"],
    "license": splade_run["license"],
    "selected_variant": selected_prefix,
    "selected_local_weight": selected_local_weight,
    "rrf_k": best_key[0],
    "splade_weight": best_key[1],
    "comparison": comparison.to_dict("records"),
    "primary_test": primary,
    "primary_look": 2,
    "primary_alpha": 0.025,
    "bootstrap_confidence": 0.975,
    "primary_confirmed": primary_confirmed,
    "secondary_tests": secondary,
    "candidate_union_recall": union_recall,
    "slice_metrics": slices.to_dict("records"),
}
(ROOT / "reports/splade_metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
)
display(answer.head())
print("answer_splade.csv validated; promoted to answer.csv:", primary_confirmed)

,slice,value,queries,BM25+dense,BM25+dense+SPLADE,delta
0,has_filters,False,1546,0.833495,0.843359,0.009864
1,has_filters,True,906,0.797829,0.800037,0.002208
2,local_pool_bucket,0,427,0.483997,0.483997,0.000000
3,local_pool_bucket,1-99,42,0.857143,0.833333,-0.023810
4,local_pool_bucket,100-499,213,0.877934,0.882629,0.004695
5,local_pool_bucket,10000+,543,0.803867,0.811848,0.007980
6,local_pool_bucket,2000-9999,554,0.921029,0.940734,0.019705
7,local_pool_bucket,500-1999,673,0.943536,0.946508,0.002972
12,positive_bucket,1,2283,0.820412,0.826982,0.006570
13,positive_bucket,2,128,0.828125,0.839844,0.011719


BM25+dense+SPLADE candidate-union Recall: 0.9443991299619358


,query_id,answer
0,70DfDUpwjxB4lzFd,fa01652ca5f57d0f 355392014208b7bf 6042fecdd775...
1,JTrdTaZJvSiLPkXj,cbeccbecb1fb8d86 8703b2a0442c5d25 16ea26ff0aa0...
2,LZCZNoVG4AFUkVRJ,dab52187b4500d9b d8fce513e4f000a7 3b370cc603f6...
3,660ac9QVtXkRxZC3,af91ec4a29b66636 bd2cb4500fad728e 67b2cd847669...
4,YgHcM9MVbxKnxD1e,1a62634394544781 13f58fe9542bdb1a 615c73ea4c3b...


answer_splade.csv validated; promoted to answer.csv: False
